# 02 · Análisis de vulnerabilidades

**Objetivo.** Describir el perfil de vulnerabilidades: severidad, identificadores
y paquetes más frecuentes, distribución por repositorio, concentración y
relaciones entre métricas.

**Entrada.** `INPUT_PATHS`, lista de reportes que se fusionan por repositorio
(SBOM + vulnerabilidades). Por defecto: `results.json` si existe; si no, todos
los que existan de `results-sbom.json` y `results-vuln.json`.

**Salida.** Ninguna escritura a disco: las figuras se muestran inline.

**Supuestos.** Si el reporte no trae vulnerabilidades ni hallazgos de CodeQL, la
sección correspondiente se declara como no disponible en lugar de fallar.

In [ ]:
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    return start


def _default_inputs(root: Path) -> list:
    scan = root / "results.json"
    if scan.is_file():
        return [scan]
    candidates = [root / "results-sbom.json", root / "results-vuln.json"]
    return [path for path in candidates if path.is_file()]


REPO_ROOT = _find_repo_root(Path.cwd())
INPUT_PATHS = _default_inputs(REPO_ROOT)
OUTPUT_DIR = REPO_ROOT / "analysis" / "outputs"

if not INPUT_PATHS:
    raise FileNotFoundError(
        "No se encontró results.json, results-sbom.json ni results-vuln.json"
    )

print("REPO_ROOT :", REPO_ROOT)
print("INPUT_PATHS:")
for path in INPUT_PATHS:
    print("  -", path)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
import io
import sys

import matplotlib

matplotlib.use("Agg")  # backend no interactivo (sin display)
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analysis import metrics
from analysis.loader import load_report, merge_reports, to_records


def show_fig(fig) -> None:
    # Muestra una figura inline usando el backend Agg (sin display).
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    display(Image(data=buffer.getvalue()))


pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

In [ ]:
reports = [load_report(path) for path in INPUT_PATHS]
report = merge_reports(reports)
records = to_records(report)
datasets = metrics.compute_datasets(report, records)
coverage = metrics.compute_coverage(report)

print("Reportes de entrada :", ", ".join(path.name for path in INPUT_PATHS))
print("Organización        :", report.organization)
print("Origen              :", report.source_kind, f"({report.source_path})")
print("Repositorios        :", len(report.repositories))
print("Vulnerabilidades    :", len(records["vulnerabilities"]))
print("Componentes SBOM    :", report.summary.get("components", 0))
print("Hallazgos CodeQL    :", len(records["findings"]))
print("Claves de datasets  :", ", ".join(datasets))

## 0. Cobertura por dimensión

`compute_coverage` expone directamente `code_coverage_ratio`,
`sbom_coverage_ratio` y `vuln_coverage_ratio` (cada uno con su dimensión como
numerador y el total de repositorios como denominador), además de `repo_failed`.
El notebook no recalcula ratios: solo los presenta.

In [ ]:
total = coverage["repositories_total"]
dimensions = pd.DataFrame(
    [
        {"dimensión": "Código (CodeQL)", "cobertura": coverage["code_coverage_ratio"]},
        {"dimensión": "SBOM", "cobertura": coverage["sbom_coverage_ratio"]},
        {"dimensión": "Grype", "cobertura": coverage["vuln_coverage_ratio"]},
    ]
)
dimensions["total"] = total
print(dimensions.to_string(index=False))
print(f"\nRepositorios fallidos (repo_failed): {coverage['repo_failed']}")

## 1. Severidad

Pregunta: *¿cómo se reparte la gravedad de las vulnerabilidades?* El peso de
High+Critical indica la urgencia de remediación.

In [ ]:
sev_df = pd.DataFrame(datasets["severity_distribution"])
total_vulns = int(sev_df["count"].sum())
sev_df["percent"] = (
    (sev_df["count"] / total_vulns * 100).round(2) if total_vulns else 0.0
)
print(sev_df.to_string(index=False))

if total_vulns:
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.bar(sev_df["severity"], sev_df["count"], color="#b2182b")
    ax.set_title("Vulnerabilidades por severidad")
    ax.set_ylabel("Vulnerabilidades")
    for x, y in zip(sev_df["severity"], sev_df["count"]):
        ax.text(x, y, str(y), ha="center", va="bottom", fontsize=8)
    show_fig(fig)
else:
    print("Sin vulnerabilidades: no se genera gráfico de severidad.")

## 2. Identificadores (CVE/GHSA) más frecuentes

Un mismo identificador puede aparecer en varios paquetes y repositorios. La
columna `repos_affected` mide su propagación.

In [ ]:
cves_df = pd.DataFrame(datasets["top_cves"])
if cves_df.empty:
    print("Sin identificadores de vulnerabilidad en el reporte.")
else:
    print(cves_df.head(15).to_string(index=False))
    top = cves_df.head(10).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(top["id"], top["count"], color="#4393c3")
    ax.set_title("Top 10 identificadores por apariciones")
    ax.set_xlabel("Apariciones")
    show_fig(fig)

## 3. Paquetes más afectados

Los paquetes con más vulnerabilidades y peor severidad son candidatos
prioritarios a actualización.

In [ ]:
pkgs_df = pd.DataFrame(datasets["top_packages"])
if pkgs_df.empty:
    print("Sin paquetes con vulnerabilidades en el reporte.")
else:
    print(pkgs_df.head(15).to_string(index=False))
    top = pkgs_df.head(10).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(top["package"], top["count"], color="#d6604d")
    ax.set_title("Top 10 paquetes por vulnerabilidades")
    ax.set_xlabel("Vulnerabilidades")
    show_fig(fig)

## 4. Distribución por repositorio

`repository_distribution` incluye todos los repositorios (también los que no
tienen vulnerabilidades), por lo que los ceros son informativos.

In [ ]:
repo_df = pd.DataFrame(datasets["repository_distribution"])
print(repo_df.to_string(index=False) if not repo_df.empty else "(sin repositorios)")

affected = repo_df[repo_df["vulnerabilities"] > 0].sort_values("vulnerabilities")
if not affected.empty:
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.barh(affected["repo"], affected["vulnerabilities"], color="#5aae61")
    ax.set_title("Vulnerabilidades por repositorio (solo afectados)")
    ax.set_xlabel("Vulnerabilidades")
    show_fig(fig)
else:
    print("Ningún repositorio presenta vulnerabilidades.")

## 5. Concentración

- `top_n_share`: fracción de vulnerabilidades en los N repositorios más
  afectados.
- `top_10pct_share`: fracción en el decil superior de repositorios afectados.
- `hhi`: Herfindahl-Hirschman (1.0 = un solo repo; tiende a 0 al repartirse).

In [ ]:
conc = datasets["concentration"]
print(f"Repositorios con vulnerabilidades : {conc['repositories_with_vulns']}")
print(f"Top N                             : {conc['top_n']}")
print(f"Cuota top N                       : {conc['top_n_share']:.2%}")
print(f"Cuota decil superior              : {conc['top_10pct_share']:.2%}")
print(f"HHI                               : {conc['hhi']:.4f}")

## 6. Relaciones entre métricas

- `fixed_version_available_share`: fracción de vulnerabilidades con corrección
  publicada.
- `components_vs_vulnerabilities`: correlación de Pearson entre componentes del
  SBOM y vulnerabilidades por repositorio. Al fusionar `results-sbom.json` con
  `results-vuln.json` esta relación pasa a ser calculable con datos reales
  (**correlación ≠ causalidad**).
- `severity_by_package_type`: severidad según el ecosistema del paquete.

In [ ]:
relations = datasets["relations"]
corr = relations["components_vs_vulnerabilities"]

print(f"Corrección disponible     : {relations['fixed_version_available_share']:.2%}")
if corr["pearson"] is None:
    print(f"Pearson componentes/vulns : no calculable (n={corr['n']})")
else:
    print(f"Pearson componentes/vulns : {corr['pearson']} (n={corr['n']})")

type_df = pd.DataFrame(relations["severity_by_package_type"])
print("\nSeveridad por tipo de paquete:")
print(type_df.to_string(index=False) if not type_df.empty else "  (sin datos)")

if corr["pearson"] is not None:
    points = pd.DataFrame(datasets["repositories"])[
        ["repo", "sbom_components", "vuln_total"]
    ]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(points["sbom_components"], points["vuln_total"], color="#2166ac")
    for _, row in points.iterrows():
        ax.annotate(
            row["repo"],
            (row["sbom_components"], row["vuln_total"]),
            fontsize=7,
            xytext=(3, 3),
            textcoords="offset points",
        )
    ax.set_title(f"Componentes vs vulnerabilidades (r={corr['pearson']})")
    ax.set_xlabel("Componentes SBOM")
    ax.set_ylabel("Vulnerabilidades")
    show_fig(fig)

## 7. Hallazgos de CodeQL

El reporte de vulnerabilidades (`miner vuln`) no incluye hallazgos de CodeQL.
Cuando faltan, se declara explícitamente en lugar de fallar.

In [ ]:
rules_df = pd.DataFrame(datasets["top_rules"])
if not records["findings"]:
    print("Este reporte no incluye hallazgos de CodeQL (findings vacío o ausente).")
    print("No pueden calcularse métricas por regla ni por lenguaje.")
else:
    print(rules_df.head(15).to_string(index=False))
    top = rules_df.head(10).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(top["rule_id"], top["count"], color="#8073ac")
    ax.set_title("Top 10 reglas de CodeQL")
    ax.set_xlabel("Hallazgos")
    show_fig(fig)

## 8. Lectura de los resultados

In [ ]:
print("Síntesis de vulnerabilidades")
print("----------------------------")
summary = report.summary
total_reported = summary.get("vulnerabilities", 0)
print(f"Vulnerabilidades (summary): {total_reported}")
if total_reported:
    high_critical = summary.get("vulns_critical", 0) + summary.get("vulns_high", 0)
    print(
        f"High+Critical: {high_critical} "
        f"({high_critical / total_reported:.1%})"
    )
    print(
        f"Con corrección disponible: "
        f"{relations['fixed_version_available_share']:.1%}"
    )
    if not pkgs_df.empty:
        leader = pkgs_df.iloc[0]
        print(f"Paquete más afectado: {leader['package']} ({leader['count']})")
else:
    print("No hay vulnerabilidades que analizar.")